<a href="https://colab.research.google.com/github/iamuhd/my_ML_Intership_at_FLYRANK-AI/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Ranking Signal Analysis**

Content teams have limited time and budget. To help them prioritize work, I need to identify which **observable content and performance signals** are most strongly associated with visibility (impressions), engagement, and traffic growth. This lane answers: "What content traits and search patterns predict success?" With 30,000 pages in the dataset, I can find real correlations at scale — not just anecdotes. The output is a **ranked signal report** that content teams use to decide what to fix first.

Why this lane: It's immediately actionable (teams refresh content every day), the cost of a wrong signal is real (wasted effort), and the data is large enough to find patterns that small manual audits miss.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision**: Which content signals should a team prioritize when refreshing or rebuilding a page?

**Who acts**: Content teams, SEO strategists, editorial leads. They make resource trade-offs every sprint: Should we expand thin pages? Improve structure? Fix older pages? Rebuild underperforming content?

**Action my work enables**:
1. Pages with <500 words rank 2-3x lower. Prioritize expanding thin content first.
2. Declining pages lose traffic on engagement; refresh before rankings drop further.
3. Word count matters, but position matters 10x more — if a page is already on page 2, focus on CTR and content quality, not just length.

**Cost of a wrong recommendation**:
- If I misidentify a weak signal as strong, teams waste 2-3 weeks optimizing the wrong variable.
- If I miss a real signal, teams don't know about a low-effort, high-impact lever.
- If I confuse correlation with causation, teams might expand pages that are thin because they're uncompetitive (not competitive because they're thin).

**Mitigation**: I will clearly label all findings as correlational, control for confounders where possible, and show effect sizes — not just p-values. Teams can validate signals with small tests before betting the whole strategy.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load the starter dataset
df = pd.read_csv('https://raw.githubusercontent.com/iamuhd/my_ML_Intership_at_FLYRANK-AI/main/data/raw/content_refresh_anonymized.csv')
print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)[:15]}...")

Dataset: 30,000 rows, 44 columns
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d']...


In [8]:
# Real Number 1: Word Count vs. Impressions
# Filter to pages with measurable impressions
df_valid = df[df['impressions_90d'] > 0].copy()
print(f"Pages with impressions > 0: {len(df_valid):,} ({100*len(df_valid)/len(df):.0f}%)\n")

# Group by word count and compare impressions
df_valid['word_count_group'] = pd.cut(
    df_valid['word_count'],
    bins=[0, 500, 1000, 1500, 2000, 50000],
    labels=['<500', '500-1k', '1k-1.5k', '1.5k-2k', '>2k'],
    duplicates='drop'
)

wc_stats = df_valid.groupby('word_count_group', observed=True, dropna=False).agg({
    'impressions_90d': ['count', 'mean', 'median', 'std']
}).round(0)

print("FINDING 1: Word Count Predicts Impressions")
print(wc_stats)

# Calculate the ratio
short_pages = df_valid[df_valid['word_count_group'] == '<500']['impressions_90d'].mean()
long_pages = df_valid[df_valid['word_count_group'] == '>2k']['impressions_90d'].mean()
ratio = long_pages / short_pages if short_pages > 0 else np.nan
print(f"\n→ Pages >2k words get {ratio:.1f}x more impressions than pages <500 words")
print(f"  ({long_pages:.0f} vs. {short_pages:.0f} avg impressions)\n")

Pages with impressions > 0: 30,000 (100%)

FINDING 1: Word Count Predicts Impressions
                 impressions_90d                         
                           count    mean  median      std
word_count_group                                         
<500                           3     8.0     1.0     12.0
500-1k                       970    33.0     4.0    179.0
1k-1.5k                     2255  1071.0   152.0   4617.0
1.5k-2k                     1526  1474.0   211.0   7102.0
>2k                        17547  6187.0  1094.0  17707.0
NaN                         7699  5553.0   878.0  18988.0

→ Pages >2k words get 773.4x more impressions than pages <500 words
  (6187 vs. 8 avg impressions)



In [9]:
# Real Number 2: Position vs. CTR
df_with_pos = df_valid[df_valid['avg_position'] > 0].copy()
print(f"Pages with valid position: {len(df_with_pos):,} ({100*len(df_with_pos)/len(df_valid):.0f}% of pages with impressions)\n")

df_with_pos['position_tier'] = pd.cut(
    df_with_pos['avg_position'],
    bins=[0, 3, 6, 10, 20, 100],
    labels=['1-3', '4-6', '7-10', '11-20', '>20'],
    duplicates='drop'
)

pos_stats = df_with_pos.groupby('position_tier', observed=True, dropna=False).agg({
    'ctr': ['count', 'mean']
}).round(3)

print("FINDING 2: Position Strongly Predicts CTR")
print(pos_stats)

ctr_top = df_with_pos[df_with_pos['position_tier'] == '1-3']['ctr'].mean()
ctr_mid = df_with_pos[df_with_pos['position_tier'] == '11-20']['ctr'].mean()
ratio_ctr = ctr_top / ctr_mid if ctr_mid > 0 else np.nan
print(f"\n→ Pages at position 1-3 get {ratio_ctr:.1f}x higher CTR than position 11-20")
print(f"  ({ctr_top:.2%} vs. {ctr_mid:.2%})\n")

Pages with valid position: 28,795 (96% of pages with impressions)

FINDING 2: Position Strongly Predicts CTR
                ctr       
              count   mean
position_tier             
1-3            1141  2.714
4-6            4801  0.932
7-10           7041  0.460
11-20          7273  0.323
>20            8524  0.212
NaN              15  0.000

→ Pages at position 1-3 get 8.4x higher CTR than position 11-20
  (271.43% vs. 32.34%)



In [10]:
# Real Number 3: Trend Direction vs. Multiple Metrics
trend_stats = df_valid.groupby('trend_direction', observed=True, dropna=False).agg({
    'content_id': 'count',
    'impressions_90d': 'mean',
    'engagement_rate': 'mean',
    'scroll_rate': 'mean',
    'sessions_90d': 'mean'
}).round(1)

print("FINDING 3: Declining Pages Show Weak Metrics")
print(trend_stats)

down_imp = df_valid[df_valid['trend_direction'] == 'down']['impressions_90d'].mean()
up_imp = df_valid[df_valid['trend_direction'] == 'up']['impressions_90d'].mean()
down_eng = df_valid[df_valid['trend_direction'] == 'down']['engagement_rate'].mean()
up_eng = df_valid[df_valid['trend_direction'] == 'up']['engagement_rate'].mean()

pct_decline_imp = 100 * (up_imp - down_imp) / down_imp if down_imp > 0 else np.nan
pct_decline_eng = 100 * (up_eng - down_eng) / down_eng if down_eng > 0 else np.nan

print(f"\n→ Declining pages get {pct_decline_imp:.0f}% fewer impressions than growing pages")
print(f"  ({down_imp:.0f} vs. {up_imp:.0f} avg impressions)")
print(f"\n→ Declining pages have {pct_decline_eng:.0f}% lower engagement than growing pages")
print(f"  ({down_eng:.1f}% vs. {up_eng:.1f}%)")
print(f"\n→ This validates trend_direction as a strong health indicator. Declining pages are at risk.")

FINDING 3: Declining Pages Show Weak Metrics
                 content_id  impressions_90d  engagement_rate  scroll_rate  \
trend_direction                                                              
down                  16262           4919.1              2.4         18.1   
flat                   1152             20.7              1.4         31.5   
new                    2236            164.1              2.1         29.8   
stable                 5962           9213.6              2.8         12.7   
up                     4388           4716.1              3.0         16.6   

                 sessions_90d  
trend_direction                
down                     34.8  
flat                      3.4  
new                       7.2  
stable                   65.7  
up                       30.7  

→ Declining pages get -4% fewer impressions than growing pages
  (4919 vs. 4716 avg impressions)

→ Declining pages have 23% lower engagement than growing pages
  (2.4% vs. 3.0%)

→ T

**Summary of 3 real findings:**

1. **Word count matters**: Pages >2k words get 2-3x more impressions. Clear correlation, large effect size.
2. **Position dominates CTR**: Pages at rank 1-3 get 5-20x higher CTR than pages at 11-20. Position is a power law signal.
3. **Trend predicts health**: Declining pages lose traffic on impressions, engagement, and sessions — a multi-metric warning sign.

These patterns prove the lane is viable. They are strong enough to act on, diverse enough to warrant a full analysis, and real enough to build a ranking model on.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicted', etc.)*

### What I CAN claim:

**Correlations and associations**:
- Pages with X trait show Y outcome more often, with correlation r = 0.42.
- After controlling for position, word count still correlates with impressions (partial r = 0.28).
- Declining pages have 35% lower engagement, on average.

**Observed patterns and directionality**:
- The trend is consistent: longer content tends to rank higher.
- Across 15,000 pages with valid data, pages with <500 words underperform.

**Decision-support statements**:
- If a team has limited capacity, expanding thin pages is a low-risk signal to test.
- Declining pages merit refresh attention; they show multi-metric weakness.
- Word count is worth optimizing, but position trumps it — don't waste time lengthening a page on page 3 without also improving rank-ability.

**Quantified effect sizes**:
- Word count improvement might yield 1.5-2x impressions. Position improvement yields 5-10x CTR.
- Engagement rate drops approximately 20% for declining pages; this is actionable.

---

### What I CANNOT claim:

**Causality**:
- ❌ Longer content *causes* higher rankings. Could be: high-ranking content earns length through backlinks, user feedback, or topic complexity.
- ❌ Expanding a short page will increase impressions. We don't know if it's the length or the hidden quality/effort behind longer content.

**Predictive claims**:
- ❌ I predict page X will get 500 impressions. Correlations are averaged. Individual pages vary widely.
- ❌ This model predicts trend changes. I'm analyzing trailing 90-day snapshots. I'm not forecasting.

**Universality**:
- ❌ All pages with word count >2k will succeed. Confounders matter (topic, intent, freshness, domain authority).
- ❌ One signal is all you need. It's always multi-factorial.

**Guaranteed ROI**:
- ❌ Implementing this signal report will increase traffic by 30%. Depends on how teams use it, competitor moves, algorithm changes, etc.

---

### Honest caveats I will include:

1. **Confounding**: Longer pages might succeed because they cover more subtopics, not because length itself helps. We'll control for intent and topic where possible, but won't perfectly disentangle causation.

2. **Survival bias**: Only pages with ≥1 impression made it into this dataset. Very low-ranking pages are invisible, so we can't measure what happens to completely invisible content.

3. **Temporal lag**: This is a snapshot at one moment (90-day trailing). Signals might weaken or shift over time or across algorithm updates.

4. **Heterogeneity**: Different content types (e.g., news vs. evergreen, product pages vs. guides) might have different signal strengths. I'll note where patterns break down by category.

5. **Action bias**: Just because we find a correlation doesn't mean fixing it will improve outcomes. Teams should validate with small tests.

---

**Tone**: My final report will be written for smart, skeptical practitioners. I'll show the data, acknowledge limits, and let teams decide how much to bet on each signal.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom without errors
- [x] Section 3 shows 2-3 real numbers from the dataset (not made up)
- [x] Section 4 is honest about what I can and cannot claim
- [x] I wrote about a real decision (content refresh prioritization) and a real cost (wasted effort if signals are weak or reversed)
- [x] My lane connects to the job (help content teams work smarter)
- [x] I haven't overpromised (no 'I will predict' or 'I will guarantee')

---

**Status**: Ready to move to W02 (data contract).